<a href="https://colab.research.google.com/github/deepthivj-aiml/5-Day-AI-Agents-Intensive-course-Google/blob/main/Copy_of_Fork_of_AutoEvalAI_AI_Powered_Second_Hand_Vehicle.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [18]:
# ==============================================
# 🔧 Install dependencies
# ==============================================
!pip install --upgrade gradio pillow --quiet

# ==============================================
# 🔑 Imports
# ==============================================
import gradio as gr
from PIL import Image, ImageEnhance, ImageFilter
import io, base64, re

# ==============================================
# ✨ Image Enhancement
# ==============================================
def enhance(img: Image.Image):
    img = img.convert("RGB")
    img = img.filter(ImageFilter.SHARPEN)
    img = ImageEnhance.Contrast(img).enhance(1.25)
    img = ImageEnhance.Color(img).enhance(1.15)
    img = ImageEnhance.Brightness(img).enhance(1.1)
    return img

def image_to_base64(img: Image.Image):
    buffered = io.BytesIO()
    img.save(buffered, format="JPEG")
    return base64.b64encode(buffered.getvalue()).decode("utf-8")

# ==============================================
# ⭐ Rating Emoji
# ==============================================
def rating_emoji(r):
    return "🤩" if r >= 8 else ("🙂" if r >= 6 else ("😐" if r >= 4 else "🙁"))

# ==============================================
# 🔎 Mock Gemini call (replace with real API)
# ==============================================
def call_gemini(prompt, img_bytes=None):
    return {
        "text": (
            "Condition: Good. Minor scratches observed. "
            "Rust: None. Paint: Well maintained. "
            "Dents: Small dent on left door. "
            "Tires: 80% tread remaining. Rims: Clean and aligned. "
            "Engine: Oil good, no leaks. Belts and hoses OK. "
            "Rating 7. ModelYear: None"
        )
    }

# ==============================================
# 🔎 ModelYear extraction / inference
# ==============================================
def extract_model_year_from_plate(license_plate):
    if not license_plate:
        return None
    match = re.search(r"(\d{2,4})", license_plate)
    if match:
        year = int(match.group(1))
        if year < 100:
            year += 2000
        return str(year)
    return None

def infer_model_year_from_features(vehicle_data):
    """
    Infer ModelYear from car features if not in plate.
    Here we use a simple rule-based approach for demonstration:
    - Make + Model + Color mapping
    """
    make = vehicle_data.get("Make", "").lower()
    model = vehicle_data.get("Model", "").lower()
    color = vehicle_data.get("Color", "").lower()

    # Example rules
    if make == "honda" and model == "civic" and color == "blue":
        return "2021"
    if make == "toyota" and model == "corolla":
        return "2020"
    return "2019"  # default fallback

# ==============================================
# 🚘 MCP + Vehicle Inspection Pipeline
# ==============================================
def analyze_mcp(fullcar, body, wheel1, wheel2, engine):

    # Mock vehicle details (simulate missing License)
    vehicle_data = {
        "Type": "Car",
        "License": "",  # Empty if no plate
        "Make": "Honda",
        "Model": "Civic",
        "Color": "Blue",
        "ModelYear": None
    }

    # Extract ModelYear from plate if exists
    vehicle_data["ModelYear"] = extract_model_year_from_plate(vehicle_data["License"])
    # If ModelYear missing, infer from features
    if not vehicle_data["ModelYear"]:
        vehicle_data["ModelYear"] = infer_model_year_from_features(vehicle_data)

    partwise_descriptions = {}
    part_ratings = []

    channels = [
        ("Full Car", fullcar, "Inspect overall exterior: dents, scratches, paint, structural issues. Rating 1–10."),
        ("Body", body, "Inspect body: rust, paint quality, dents, alignment. Rating 1–10."),
        ("Wheel 1", wheel1, "Inspect wheel: tyre wear, rim, cracks, alignment. Rating 1–10."),
        ("Wheel 2", wheel2, "Inspect wheel: tyre wear, rim, cracks, alignment. Rating 1–10."),
        ("Engine", engine, "Inspect engine: leaks, rust, oil, belts, hoses. Rating 1–10.")
    ]

    for title, file, prompt_text in channels:
        if file is None:
            partwise_descriptions[title] = "❌ Not uploaded."
            continue

        # Convert to PIL and enhance
        if not isinstance(file, Image.Image):
            import numpy as np
            file = Image.fromarray(file)
        img = enhance(file)
        img_base64 = image_to_base64(img)

        # Call Gemini (mock)
        res = call_gemini(prompt_text, img_bytes=img_base64)
        text = res.get("text", "")

        # Extract rating
        rating_match = re.search(r"Rating (\d+(\.\d+)?)", text)
        rating = float(rating_match.group(1)) if rating_match else 6
        part_ratings.append(rating)
        emoji = rating_emoji(rating)

        partwise_descriptions[title] = f"{text} {emoji}"

    # Average rating
    avg_rating = sum(part_ratings)/len(part_ratings) if part_ratings else 0
    avg_emoji = rating_emoji(avg_rating)

    # ===================== HTML Output =====================
    html = f"""
    <div style='padding:12px; background:#e6f0ff; border-radius:12px;'>
        <h2 style='color:#003366;'>🗓 Vehicle Details</h2>
        <ul style='font-size:16px; color:#000;'>
            <li><b>Type:</b> {vehicle_data['Type']}</li>
            <li><b>License:</b> {vehicle_data['License'] or 'N/A'}</li>
            <li><b>Make:</b> {vehicle_data['Make']}</li>
            <li><b>Model:</b> {vehicle_data['Model']}</li>
            <li><b>Color:</b> {vehicle_data['Color']}</li>
            <li><b>Model Year:</b> {vehicle_data['ModelYear']}</li>
        </ul>
        <h2 style='color:#003366;'>⭐ Average Rating: {avg_rating:.1f} {avg_emoji}</h2>
    </div>
    <hr style='border:1px solid #003366;'/>
    """

    html += "<h2 style='color:#003366;'>🔍 Part-wise Inspection</h2>"
    for part, desc in partwise_descriptions.items():
        html += f"""
        <div style='padding:10px; margin:6px; background:#cce0ff; border-radius:10px;'>
            <h3 style='color:#001f4d;'>{part}</h3>
            <p style='color:black;'>{desc}</p>
        </div>
        """

    return html

# ==============================================
# 🎨 Gradio Blocks UI
# ==============================================
CSS = """
<style>
body { background: #b3d1ff !important; }
.gradio-container { background: #b3d1ff !important; }
* { color: black !important; font-family:Arial, sans-serif; }
</style>
"""

with gr.Blocks() as demo:

    gr.HTML(CSS)
    gr.Markdown("<h2 style='text-align:center;color:#003366;'>🚘 Used Car Multi-Part Inspection</h2>")
    gr.Markdown("### Upload Full Car + Individual Parts")

    with gr.Row():
        fullcar = gr.Image(label="Full Car Image", type="numpy")
        body = gr.Image(label="Body Image", type="numpy")

    with gr.Row():
        wheel1 = gr.Image(label="Wheel 1 Image", type="numpy")
        wheel2 = gr.Image(label="Wheel 2 Image", type="numpy")

    engine = gr.Image(label="Engine Image", type="numpy")

    run_btn = gr.Button("Run Inspection")
    output = gr.HTML()

    run_btn.click(
        analyze_mcp,
        [fullcar, body, wheel1, wheel2, engine],
        output
    )

# Launch continuously until manual interrupt
demo.launch(share=True, prevent_thread_lock=True)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7.0 MB 29.1 MB/s eta 0:00:00
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
Running on public URL: https://1cf4a14ce0cf83b419.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from Terminal to deploy to Spaces (https://huggingface.co/spaces)
